# Customer Shopping Behavior Analysis

Notebook for customer shopping behavior analysis with modular helpers and additional insights.

In [1]:
import pandas as pd

# Load source dataset
raw_data_path = "../data/raw/customer_shopping_behavior.csv"
shopping_df = pd.read_csv(raw_data_path)

shopping_df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [2]:
# Quick schema check
shopping_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   str    
 3   Item Purchased          3900 non-null   str    
 4   Category                3900 non-null   str    
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   str    
 7   Size                    3900 non-null   str    
 8   Color                   3900 non-null   str    
 9   Season                  3900 non-null   str    
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   str    
 12  Shipping Type           3900 non-null   str    
 13  Discount Applied        3900 non-null   str    
 14  Promo Code Used         3900 non-null   str    
 15

In [3]:
# Full descriptive profile
shopping_df.describe(include="all")

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


In [4]:
# Missing-value scan
shopping_df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [5]:
def fill_review_rating_by_category_median(input_df: pd.DataFrame) -> pd.DataFrame:
    """Impute missing review ratings with category-level median ratings."""
    output_df = input_df.copy()
    output_df["Review Rating"] = output_df.groupby("Category")["Review Rating"].transform(
        lambda rating_series: rating_series.fillna(rating_series.median())
    )
    return output_df


shopping_df = fill_review_rating_by_category_median(shopping_df)
shopping_df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [6]:
def standardize_column_names(input_df: pd.DataFrame) -> pd.DataFrame:
    """Convert column names to snake_case and normalize purchase amount column."""
    renamed_df = input_df.copy()
    renamed_df.columns = (
        renamed_df.columns
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    renamed_df = renamed_df.rename(columns={"purchase_amount_(usd)": "purchase_amount"})
    return renamed_df


shopping_df = standardize_column_names(shopping_df)
shopping_df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='str')

In [7]:
def add_age_group_segment(input_df: pd.DataFrame) -> pd.DataFrame:
    """Create quartile-based age segments."""
    segmented_df = input_df.copy()
    age_group_labels = ["Young Adult", "Adult", "Middle-aged", "Senior"]
    segmented_df["age_group"] = pd.qcut(segmented_df["age"], q=4, labels=age_group_labels)
    return segmented_df


shopping_df = add_age_group_segment(shopping_df)
shopping_df[["age", "age_group"]].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [8]:
def map_purchase_frequency_to_days(input_df: pd.DataFrame) -> pd.DataFrame:
    """Map frequency labels to numeric days for easier quantitative analysis."""
    converted_df = input_df.copy()
    frequency_to_days_map = {
        "Fortnightly": 14,
        "Weekly": 7,
        "Monthly": 30,
        "Quarterly": 90,
        "Bi-Weekly": 14,
        "Annually": 365,
        "Every 3 Months": 90,
    }
    converted_df["purchase_frequency_days"] = converted_df["frequency_of_purchases"].map(frequency_to_days_map)
    return converted_df


shopping_df = map_purchase_frequency_to_days(shopping_df)
shopping_df[["purchase_frequency_days", "frequency_of_purchases"]].head(10)

,purchase_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [9]:
# Validate discount and promo-code consistency
shopping_df[["discount_applied", "promo_code_used"]].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [10]:
(shopping_df["discount_applied"] == shopping_df["promo_code_used"]).all()

np.True_

In [11]:
def drop_redundant_promo_code_column(input_df: pd.DataFrame) -> pd.DataFrame:
    """Drop promo_code_used because it duplicates discount_applied information."""
    return input_df.drop("promo_code_used", axis=1)


shopping_df = drop_redundant_promo_code_column(shopping_df)
shopping_df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='str')

## Extra Insights

In [12]:
# Insight 1: Revenue by category
category_revenue = (
    shopping_df.groupby("category", as_index=False)["purchase_amount"]
    .sum()
    .sort_values("purchase_amount", ascending=False)
)
category_revenue.head(10)

,category,purchase_amount
1,Clothing,104264
0,Accessories,74200
2,Footwear,36093
3,Outerwear,18524


In [13]:
# Insight 2: Average rating by season
season_rating_summary = (
    shopping_df.groupby("season", as_index=False)["review_rating"]
    .mean()
    .sort_values("review_rating", ascending=False)
)
season_rating_summary

,season,review_rating
1,Spring,3.788388
3,Winter,3.750463
0,Fall,3.732718
2,Summer,3.727225


In [14]:
# Insight 3: High-value customer segments by age group and subscription status
segment_spend_summary = (
    shopping_df.groupby(["age_group", "subscription_status"], as_index=False)["purchase_amount"]
    .mean()
    .sort_values("purchase_amount", ascending=False)
)
segment_spend_summary.head(10)

,age_group,subscription_status,purchase_amount
1,Young Adult,Yes,60.456929
0,Young Adult,No,60.448095
4,Middle-aged,No,60.162393
5,Middle-aged,Yes,59.728873
2,Adult,No,59.468023
6,Senior,No,59.320402
3,Adult,Yes,59.307087
7,Senior,Yes,58.370968


## Export to SQL Databases

In [15]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [16]:
def export_to_postgresql(input_df: pd.DataFrame, username: str, password: str, host: str, port: str, database: str, table_name: str = "customer") -> None:
    engine = create_engine(f"postgresql+psycopg2://{username}:{password}@{host}:{port}/{database}")
    input_df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Data loaded into table '{table_name}' in database '{database}'.")


# Example usage
# export_to_postgresql(shopping_df, "postgres", "your_password", "localhost", "5432", "customer_behavior")

In [17]:
def export_to_mysql(input_df: pd.DataFrame, username: str, password: str, host: str, port: str, database: str, table_name: str = "customer") -> None:
    engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")
    input_df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Data loaded into MySQL table '{table_name}' in database '{database}'.")


# Example usage
# export_to_mysql(shopping_df, "root", "your_password", "localhost", "3306", "customer_behavior")

In [18]:
def export_to_sql_server(input_df: pd.DataFrame, username: str, password: str, host: str, port: str, database: str, driver_name: str = "ODBC Driver 17 for SQL Server", table_name: str = "customer") -> None:
    encoded_driver = quote_plus(driver_name)
    engine = create_engine(
        f"mssql+pyodbc://{username}:{password}@{host},{port}/{database}?driver={encoded_driver}"
    )
    input_df.to_sql(table_name, engine, if_exists="replace", index=False)
    print(f"Data loaded into SQL Server table '{table_name}' in database '{database}'.")


# Example usage
# export_to_sql_server(shopping_df, "sa", "your_password", "localhost", "1433", "customer_behavior")

In [19]:
# Final prepared data snapshot
shopping_df.head()

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,previous_purchases,payment_method,frequency_of_purchases,age_group,purchase_frequency_days
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,14,Venmo,Fortnightly,Middle-aged,14
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,2,Cash,Fortnightly,Young Adult,14
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,23,Credit Card,Weekly,Middle-aged,7
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,49,PayPal,Weekly,Young Adult,7
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,31,PayPal,Annually,Middle-aged,365
